Processamento distribuído com PySpark

### Ciclo 1 — SparkSession e DataFrame

Identificar a versão do Spark, criar um DataFrame manualmente e exibi-lo.

In [ ]:
# Databricks disponibiliza a sessão Spark como variável spark
print("Versão do Spark:", spark.version)

df = spark.createDataFrame(
    [(1, "um"), (2, "dois"), (3, "três"), (4, "quatro"), (5, "cinco")],
    ["id", "nome"]
)

print("DataFrame de exemplo:")
df.show()

**Atividade 1 — Criar um DataFrame**

Crie um DataFrame `df_produtos` com duas colunas (`produto`, `preco`) e os registros **Caderno / 25.0**, **Mouse / 65.0** e **Teclado / 85.0**. Exiba com `.show()`.

In [ ]:
# Atividade 1
# TODO: crie df_produtos com as colunas produto e preco.
# Registros: Caderno 25.0; Mouse 65.0; Teclado 85.0.
# Exiba com show().

### Ciclo 2 — Transformações em DataFrames

Seleção, filtro, média por grupo, coluna condicional e ordenação.

In [ ]:
from pyspark.sql.functions import col, avg, max as max_, min as min_, count, when

# Criação de um DataFrame de exemplo
dados = [
    ("João", "M", 28, 3000),
    ("Maria", "F", 34, 4600),
    ("Carlos", "M", 23, 2500),
    ("Ana", "F", 45, 5200),
    ("Paula", "F", 29, 3100),
    ("Rafael", "M", 38, 4700)
]

colunas = ["nome", "sexo", "idade", "salario"]

df = spark.createDataFrame(dados, colunas)

# Exibe os dados originais
print("DataFrame original:")
df.show()

In [ ]:
# Seleciona apenas nome e salário
print("Seleção de colunas:")
df.select("nome", "salario").show()

In [ ]:
# Filtra pessoas com salário acima de 4000
print("Filtrando salário > 4000:")
df.filter(col("salario") > 4000).show()

In [ ]:
# Calcula média de salário por sexo
print("Média salarial por sexo:")
df.groupBy("sexo").agg(avg("salario").alias("media_salarial")).show()

In [ ]:
# Adiciona coluna com faixa etária
print("Adicionando coluna de faixa etária:")
df_com_faixa = df.withColumn(
    "faixa_etaria",
    when(col("idade") < 30, "Jovem")
    .when(col("idade") < 40, "Adulto")
    .otherwise("Sênior")
)

df_com_faixa.show()

In [ ]:
# Ordena pelo salário (descendente)
print("Ordenação por salário decrescente:")
df.orderBy(col("salario").desc()).show()

# Copia o cojunto de dados para df_funcionarios
df_funcionarios = df

**Atividade 2 — Seleção, filtro e ordenação**

A partir de `df_funcionarios`, exiba apenas `nome`, `idade` e `salario` dos funcionários com **30 anos ou mais**, ordenando pelo maior salário. Repita as operações demonstradas.

In [ ]:
# Atividade 2
# TODO: a partir de df_funcionarios, filtre idade >= 30.
# Exiba nome, idade e salario em ordem decrescente de salario.

### Ciclo 3 — Spark SQL

Demonstração com `createOrReplaceTempView()`, `SELECT`, `WHERE`, `AVG`, `GROUP BY`, `ROUND` e `CASE`.

In [ ]:
# Criando um DataFrame de exemplo
dados = [
    ("João", "M", 28, 3000),
    ("Maria", "F", 34, 4600),
    ("Carlos", "M", 23, 2500),
    ("Ana", "F", 45, 5200),
    ("Paula", "F", 29, 3100),
    ("Rafael", "M", 38, 4700)
]
colunas = ["nome", "sexo", "idade", "salario"]

df = spark.createDataFrame(dados, colunas)

# Registrando a tabela temporária
df.createOrReplaceTempView("funcionarios")

In [ ]:
# Consulta 1: Todos os dados
print("Consulta SQL: SELECT * FROM funcionarios")
spark.sql("SELECT * FROM funcionarios").show()

In [ ]:
# Consulta 2: Funcionários com salário > 4000
print("Funcionários com salário acima de 4000")
spark.sql("SELECT nome, salario FROM funcionarios WHERE salario > 4000").show()

In [ ]:
# Consulta 3: Média salarial por sexo
print("Média salarial por sexo")
spark.sql("""
    SELECT sexo, ROUND(AVG(salario), 2) as media_salarial
    FROM funcionarios
    GROUP BY sexo
""").show()

In [ ]:
# Consulta 4: Classificação por idade (usando CASE)
print("Faixa etária por SQL")
spark.sql("""
    SELECT nome, idade,
        CASE
            WHEN idade < 30 THEN 'Jovem'
            WHEN idade < 40 THEN 'Adulto'
            ELSE 'Sênior'
        END AS faixa_etaria
    FROM funcionarios
""").show()

**Atividade 3 — Média de idade por sexo**

A partir da view temporária `funcionarios`, use `spark.sql()` para calcular a **idade média por sexo**, com duas casas decimais. Adapte a consulta de média salarial demonstrada.

In [ ]:
# Atividade 3
# TODO: use spark.sql() para obter ROUND(AVG(idade), 2) por sexo.
# Consulte a view temporária funcionarios.

### Ciclo 4 — Estatística descritiva

Demonstração com `describe()`, médias, desvio-padrão, mínimo, máximo, correlação e agregações por grupo.

In [ ]:
from pyspark.sql.functions import col, mean, stddev, min as min_, max as max_, corr, count

# DataFrame de exemplo
dados = [
    (1, "A", 10.0, 100.0),
    (2, "A", 12.0, 110.0),
    (3, "B", 15.0, 105.0),
    (4, "B", 14.0, 115.0),
    (5, "C", 11.0, 120.0)
]

colunas = ["id", "grupo", "nota1", "nota2"]
df = spark.createDataFrame(dados, colunas)

# Exibe os dados
print("📋 DataFrame de entrada:")
df.show()

In [ ]:
# Estatísticas descritivas gerais
print("Estatísticas descritivas com .describe():")
df.describe(["nota1", "nota2"]).show()

In [ ]:
# Estatísticas com funções explícitas
print("Estatísticas específicas:")
df.select(
    mean("nota1").alias("média_nota1"),
    stddev("nota1").alias("desvio_padrao_nota1"),
    min_("nota2").alias("mínimo_nota2"),
    max_("nota2").alias("máximo_nota2")
).show()

In [ ]:
# Correlação entre colunas
print("Correlação entre nota1 e nota2:")
df.select(corr("nota1", "nota2").alias("correlação")).show()

In [ ]:
# Estatísticas por grupo
print("Estatísticas agrupadas por 'grupo':")
df.groupBy("grupo").agg(
    count("*").alias("total"),
    mean("nota1").alias("média_nota1"),
    mean("nota2").alias("média_nota2")
).show()

**Atividade 4 — Indicadores por grupo**

A partir do DataFrame `df` de notas, agrupe por `grupo` e calcule: **total de registros**, **média de `nota2`** e **máximo de `nota1`**. Exiba o resultado com `.show()`.

In [ ]:
# Atividade 4
# TODO: a partir de df, agrupe por grupo e calcule:
# quantidade de registros, média de nota2 e máximo de nota1.

## Ciclo 5 — Particionamento e processamento distribuído

O conjunto de dados de demonstração será repartido visualizar esses conceitos.

In [ ]:
from pyspark.sql.functions import spark_partition_id

# repartition() é uma transformação: o Spark prepara o processamento
df_particionado = df_funcionarios.repartition(2)

# O número da partição não representa uma máquina nem um identificador fixo
df_particionado = df_particionado.withColumn(
    "particao", spark_partition_id()
)

# groupBy() organiza a contagem; show() é a ação que solicita os resultados
print("Funcionários por partição:")
df_particionado.groupBy("particao").count().orderBy("particao").show()

# O Spark pode precisar de um shuffle para reunir registros de um mesmo sexo
print("Plano físico da contagem por sexo:")
df_particionado.groupBy("sexo").count().explain(mode="formatted")

## Desafio final — Análise de pedidos de uma distribuidora

**Contexto:** uma distribuidora quer acompanhar os pedidos **em aberto** e entender quais prioridades concentram mais solicitações e maiores valores.

**Fonte:** `samples.tpch.orders`, uma tabela de exemplo do próprio Databricks, pertencente ao esquema **TPC-H**. A base já está disponível no catálogo.

A próxima célula **prepara a base**: lê até 5.000 pedidos, atribui nomes em português às colunas e traduz as situações e prioridades. Não é necessário modificá-la.

In [ ]:
# Base de exemplo oficial do Databricks
from pyspark.sql.functions import col, when

pedidos = (
    spark.table('samples.tpch.orders')
    .select(
        col('o_orderkey').alias('codigo_pedido'),
        col('o_orderstatus').alias('situacao'),
        col('o_orderpriority').alias('prioridade'),
        col('o_totalprice').alias('valor_pedido')
    )
    .limit(5000)
)

pedidos = (
    pedidos
    .withColumn(
        'situacao',
        when(col('situacao') == 'O', 'Em aberto')
        .when(col('situacao') == 'F', 'Finalizado')
        .when(col('situacao') == 'P', 'Parcial')
        .otherwise(col('situacao'))
    )
    .withColumn(
        'prioridade',
        when(col('prioridade') == '1-URGENT', '1 - Urgente')
        .when(col('prioridade') == '2-HIGH', '2 - Alta')
        .when(col('prioridade') == '3-MEDIUM', '3 - Média')
        .when(col('prioridade') == '4-NOT SPECIFIED', '4 - Não definida')
        .when(col('prioridade') == '5-LOW', '5 - Baixa')
        .otherwise(col('prioridade'))
    )
)

print('Pedidos disponíveis para o desafio:', pedidos.count())
pedidos.show(6, truncate=False)

### **Tarefa**

A partir do DataFrame `pedidos`, já preparado acima:

1. Filtre os registros com `situacao == "Em aberto"`.
2. Agrupe por `prioridade`.
3. Calcule a **quantidade de pedidos** e o **valor médio dos pedidos** em cada prioridade.
4. Ordene pela maior quantidade de pedidos e exiba a tabela com `.show()`.
5. Registre o resultado como view temporária `resumo_prioridades` e consulte-a usando `spark.sql()`.
6. Use `.explain(mode="formatted")` para observar o plano de execução da agregação.

In [ ]:
# Desafio final
# TODO: filtre pedidos em aberto e agrupe por prioridade.
# Calcule total de pedidos e valor médio, ordene e exiba.
# Crie a view resumo_prioridades, consulte por SQL e visualize explain().